# TEK MODEL — e5-large DOGRUDAN-HEDEF full fine-tune (`e5ft`)

**Tez (kullanici):** `e5_ridge` (frozen embed+Ridge, rw 158) ve `txt_ridge` (TF-IDF+Ridge, rw 168)
ikisi de AYNI metni (`mentor_feedback_text`) frozen isliyor. Bunlari 2 ayri model yerine TEK bir
**fine-tune edilmis transformer** ile degistir: govde + head birlikte, dogrudan career_success_score'a.
Fine-tune frozen+TF-IDF'ten guclu (fezadangelenler atilimi); tek model ikisinin isini yapar + ortogonal.

**Model:** `intfloat/multilingual-e5-large` (1024-dim, 560M) — bizim frozen e5 ile AYNI taban, ama
FROZEN degil FINE-TUNE. (Alternatif: `xlm-roberta-large` veya `dbmdz/bert-base-turkish-cased`.)

**Cikti:** `oof_e5ft.npy` + `test_e5ft.npy` (clip[0,100], student_id sirasi, repeat-0).
Indir -> repo `artifacts/`. Lokalde: `python src/new_model_gate.py e5ft` (nested paired-test + blend etkisi).

**Kullanim:** 1. Runtime -> GPU (A100/T4). 2. Yukle: `train.csv`, `test_x.csv`, `folds.parquet`.
3. Sirayla calistir (~45dk A100 / ~2.5sa T4). 4. Inen 2 .npy'yi `artifacts/`'a koy.

**DETERMINIZM:** SEED=42; neural/GPU/bf16 -> mm/xlmr gibi 'belgelenmis tolerans'.
Karar metrigi LOKALDE nested rw-OOF + paired-test + public-gap (Colab standalone yalniz sensor).

In [ ]:
!pip -q install transformers==4.44.2 accelerate sentencepiece pyarrow

In [ ]:
from google.colab import files
up = files.upload()  # YUKLE: train.csv, test_x.csv, folds.parquet

In [ ]:
# === SABITLER + KANONIK VERI (src/cv.py ile BIREBIR) ===
import numpy as np, pandas as pd
SEED = 42
TARGET = "career_success_score"; TEXT = "mentor_feedback_text"; ID = "student_id"
RECENCY_COL = "graduation_year"; CLIP_LO, CLIP_HI = 0.0, 100.0

train = pd.read_csv("train.csv", encoding="utf-8-sig")
test  = pd.read_csv("test_x.csv", encoding="utf-8-sig")
assert train[ID].iloc[0] == "STU_000001" and test[ID].iloc[0] == "STU_010001"
assert len(train) == 10000 and len(test) == 10000
y = train[TARGET].values.astype(np.float64)
# e5 'query: ' prefix (asimetrik encoder standardi; frozen e5'le tutarli temsil)
txt_tr = ["query: " + ("" if t is None else str(t)) for t in train[TEXT].values]
txt_te = ["query: " + ("" if t is None else str(t)) for t in test[TEXT].values]
assert not any(any(ch.isdigit() for ch in t) for t in train[TEXT].astype(str)), "metinde rakam var."

In [ ]:
# === folds.parquet repeat-0 (xlmr/mm notebook ile BIREBIR) ===
folds = pd.read_parquet("folds.parquet")
pos = {sid: i for i, sid in enumerate(train[ID].values)}
fr0 = folds[folds["repeat"] == 0]
fold_of = np.full(len(train), -1, dtype=np.int64)
for sid, f in zip(fr0["student_id"].values, fr0["fold"].values): fold_of[pos[sid]] = int(f)
assert (fold_of >= 0).all()
N_SPLITS = int(fold_of.max()) + 1
FOLDS = [(np.where(fold_of != f)[0], np.where(fold_of == f)[0]) for f in range(N_SPLITS)]
print(f"repeat-0: {N_SPLITS} fold, val = {[len(v) for _,v in FOLDS]}")

In [ ]:
# === e5-large DOGRUDAN-HEDEF full fine-tune (govde+head birlikte; FROZEN DEGIL) ===
import torch, torch.nn as nn, random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

MODEL_NAME = "intfloat/multilingual-e5-large"   # bizim frozen e5 ile AYNI taban, ama fine-tune
MAX_LEN, BATCH, EPOCHS = 192, 16, 3
LR_BERT, LR_HEAD = 1e-5, 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP = torch.bfloat16 if USE_BF16 else torch.float16
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("device:", DEVICE, "| model:", MODEL_NAME, "| amp:", AMP)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextData(Dataset):
    def __init__(self, texts, tgt=None, ym=0.0, ys=1.0):
        self.t=texts; self.y=tgt; self.ym=ym; self.ys=ys
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = tok(self.t[i], truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        it = {"input_ids": e["input_ids"].squeeze(0), "attention_mask": e["attention_mask"].squeeze(0)}
        if self.y is not None:
            it["label"] = torch.tensor((self.y[i]-self.ym)/self.ys, dtype=torch.float32)
        return it

class E5Reg(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME); h = self.bert.config.hidden_size
        self.head = nn.Sequential(nn.Linear(h,256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256,1))
    def forward(self, input_ids, attention_mask):
        hs = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        # e5 standardi: mean-pooling (frozen e5'le AYNI temsil) -> head
        mean = (hs*m).sum(1)/m.sum(1).clamp(min=1e-9)
        return self.head(mean).squeeze(-1)

@torch.no_grad()
def predict(model, texts, ym, ys):
    model.eval(); out=[]
    for b in DataLoader(TextData(texts), batch_size=64):
        ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=AMP): p = model(ids, am)
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)*ys + ym

oof = np.zeros(len(train), dtype=np.float64); test_pred = np.zeros(len(test), dtype=np.float64)
for f,(tr_idx, va_idx) in enumerate(FOLDS):
    print(f"\n== Fold {f+1}/{N_SPLITS} ==")
    ym, ys = float(y[tr_idx].mean()), float(y[tr_idx].std())   # FOLD-SAFE hedef z-score
    model = E5Reg().to(DEVICE)
    dl = DataLoader(TextData([txt_tr[i] for i in tr_idx], y[tr_idx], ym, ys),
                    batch_size=BATCH, shuffle=True, drop_last=True)
    bp=[p for n,p in model.named_parameters() if n.startswith("bert.")]
    hp=[p for n,p in model.named_parameters() if not n.startswith("bert.")]
    opt = torch.optim.AdamW([{"params":bp,"lr":LR_BERT},{"params":hp,"lr":LR_HEAD}], weight_decay=0.01)
    tot = len(dl)*EPOCHS; sch = get_linear_schedule_with_warmup(opt, int(0.1*tot), tot)
    scaler = torch.cuda.amp.GradScaler(enabled=not USE_BF16)
    for ep in range(EPOCHS):
        model.train()
        for b in dl:
            opt.zero_grad()
            ids=b["input_ids"].to(DEVICE); am=b["attention_mask"].to(DEVICE); lab=b["label"].to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=AMP):
                loss = nn.functional.mse_loss(model(ids, am), lab)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        vp = np.clip(predict(model, [txt_tr[i] for i in va_idx], ym, ys), CLIP_LO, CLIP_HI)
        print(f"   epoch {ep+1} val MSE={np.mean((y[va_idx]-vp)**2):.4f}")
    oof[va_idx] = np.clip(predict(model, [txt_tr[i] for i in va_idx], ym, ys), CLIP_LO, CLIP_HI)
    test_pred  += np.clip(predict(model, txt_te, ym, ys), CLIP_LO, CLIP_HI) / N_SPLITS
    del model; torch.cuda.empty_cache()

oof = np.clip(oof, CLIP_LO, CLIP_HI); test_pred = np.clip(test_pred, CLIP_LO, CLIP_HI)
print(f"\n=== e5ft unweighted-OOF (repeat-0) = {np.mean((y-oof)**2):.4f} ===")

In [ ]:
# === standalone rw-OOF (frozen e5 158.46 / txt_ridge 168.02 ile kiyas) ===
tr_g = train[RECENCY_COL].value_counts(normalize=True); te_g = test[RECENCY_COL].value_counts(normalize=True)
w = (train[RECENCY_COL].map(te_g).fillna(0.0)/train[RECENCY_COL].map(tr_g)).to_numpy(float); w=w/w.mean()
rw = float(np.sum(w*(y-oof)**2)/np.sum(w))
print(f"e5ft standalone:  unweighted-OOF={np.mean((y-oof)**2):.4f}   recency-weighted-OOF={rw:.4f}")
print("  (frozen e5_ridge 158.46 / txt_ridge 168.02 / fine-tune xlmr 140.72 / mm 94.82)")
print("  Fine-tune frozen'dan DUSUK rw bekleriz (xlmr 140 seviyesi). NIHAI KARAR lokal:")
print("  src/new_model_gate.py e5ft -> nested paired-test + blend etkisi + corr(e5/xlmr/txt).")
print("  corr(e5_ridge/txt_ridge) DUSUKSE: tek model ikisini ikame eder + ortogonal yeni sinyal.")

In [ ]:
# === artefakt yaz + indir ===
assert oof.shape == (len(train),) and test_pred.shape == (len(test),)
assert np.isfinite(oof).all() and np.isfinite(test_pred).all()
assert oof.min()>=0 and oof.max()<=100 and test_pred.min()>=0 and test_pred.max()<=100
np.save("oof_e5ft.npy", oof.astype(np.float64)); np.save("test_e5ft.npy", test_pred.astype(np.float64))
print("YAZILDI: oof_e5ft.npy", oof.shape, "| test_e5ft.npy", test_pred.shape)
from google.colab import files
files.download("oof_e5ft.npy"); files.download("test_e5ft.npy")